# Compute-Matched Encoders vs. Frozen Pathology Foundation Model
## with Calibration, Selective Prediction, and BACH OOD on Breast Histopathology

Single Kaggle GPU notebook implementing five combined contributions:

1. **Compute-matched Pareto** — CNN, Swin, and Mamba-family encoders trained from scratch vs. a frozen public pathology foundation model (Phikon), evaluated at matched inference FLOPs on patient-disjoint leave-one-magnification-out (LOMO) BreakHis.
2. **Cross-magnification calibration benchmark** — ECE, Brier, temperature scaling across all methods.
3. **Selective prediction** — risk-coverage curves and risk@coverage-{80, 90, 95}.
4. **BACH as cross-institution OOD test** — MSP and Energy scoring.
5. **Reproducibility artifact** — resumable, incremental-save, split-manifest-hashed pipeline.

**Mamba strategy:** MedMamba/DSA-Mamba use the compiled `mamba_ssm` selective-scan kernel, which fails on newer Kaggle Torch/CUDA images. This notebook uses **MambaVision-Tiny-1K** (`timm.mambavision_tiny_1k`) as a pip-installable Mamba proxy so the state-space-vs-attention comparison always runs. MedMamba/DSA-Mamba are additionally attempted; both are optional.

Attach public Kaggle datasets, enable GPU + Internet, then run every cell in order.

## Team run guide (Kaggle only — no local machine setup needed)

Each teammate uses their own Kaggle account.

### One-time per teammate
1. Log in at https://www.kaggle.com.
2. **File > Import Notebook** (upload this `.ipynb`) or start a new notebook and paste cells.
3. Right sidebar **Settings**:
   - Accelerator: **GPU T4 x1**
   - Internet: **ON**
   - Persistence: **OFF**
4. Right sidebar **Data > Add Input**, search and **+** attach these two public datasets (no HF token, no registration):
   - `ambarish/breakhis`  — https://www.kaggle.com/datasets/ambarish/breakhis
   - `truthisneverlinear/bach-breast-cancer-histology-images`  — https://www.kaggle.com/datasets/truthisneverlinear/bach-breast-cancer-histology-images
5. In cell-2 set **RUN_NAME** to your own tag (e.g. `'abid_day1'`, `'teammate_a_day1'`).
6. Keep `RUN_MODE = 'quick_12h'` for the main 12-hour run. **Run All**.
7. Cell-13 prints a ZIP path (`/kaggle/working/<RUN_NAME>.zip`). Download it.

### RUN_MODE presets

| Mode | Scope | Expected wall-clock on Kaggle T4 |
| --- | --- | --- |
| `'pilot'` | 1 mag × 1 fraction × 1 seed × all active models | 30–60 min |
| `'quick_12h'` (default) | 4 mags × 1 fraction (1.0) × 3 seeds × CNN+Swin+MambaVision+Phikon | 10–11 h |
| `'medium_12h'` | 2 mags × 3 fractions × 5 seeds; split MAGS across teammates | 10–11 h per teammate |
| `'full'` | 4 mags × 3 fractions × 5 seeds | 40+ h total (multi-session) |

### Resumability

Every finished run writes its own `record.json` and appends to `raw_runs.csv` immediately. If the Kaggle session times out or crashes, restart the notebook with the same `RUN_NAME`: cell-11 skips runs whose `record.json` already exists and continues from where it stopped. Cell-11 also aborts cleanly 15 min before `MAX_RUNTIME_HOURS` (default 11.5).

### If something fails

- **Cell-3 `mamba_ssm` install fails**: safe to continue — MedMamba/DSA-Mamba are skipped; MambaVision through `timm` still provides the Mamba axis.
- **Cell-3 MambaVision unavailable in installed `timm`**: cell-6 removes it from the active model list; ConvNeXt + Swin + Phikon still run.
- **Cell-5 dataset path auto-discovery fails**: check the dataset is actually attached. As a last resort, hard-set `BREAKHIS_ROOT = '/kaggle/input/breakhis'` in cell-2.
- **Any other error**: copy the cell code + full traceback and share — do not silently patch.

In [ ]:
# Immutable protocol. Change only before any training run has started; then archive config.json.
RUN_MODE = 'quick_12h'                     # 'pilot' | 'quick_12h' | 'medium_12h' | 'full'
RUN_NAME = None             # None = derived from the held-out magnifications, e.g. main_ho40
HELD_OUT_OVERRIDE = [400]

# Leave these as None to auto-discover under /kaggle/input. Override only if discovery fails.
BREAKHIS_ROOT = None
BACH_ROOT = None

# OUTPUT is set further down, once RUN_NAME is resolved.
SOURCE_ROOT = '/kaggle/working/mamba_sources'
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TASK = 'binary'
NUM_CLASSES = 2

# From-scratch encoders that this pipeline knows how to build. MambaVision is a pip-installable Mamba proxy;
# MedMamba and DSA-Mamba are appended by cell-4 only if the compiled mamba_ssm kernel is available.
# dsamamba is appended by the sources cell once its import succeeds; listing it here
# keeps the intent explicit. Verified working after restoring the upstream
# model/cross_attention.py (43.40M params, ~4.2 GiB at batch 32 on a T4).
FROM_SCRATCH_MODELS = ['convnext_tiny', 'swin_tiny', 'mambavision_tiny_1k']

FM_NAME = 'owkin/phikon'                   # public; no HF token needed
USE_FOUNDATION_MODEL = True

# Per-profile knobs. Do not edit these directly; change RUN_MODE above.
# MAGNIFICATIONS is the POOL the LOMO split draws training data from and must always
# list every magnification present in the dataset. HELD_OUT_MAGS says which held-out
# values THIS session iterates, so the work can be split across Kaggle sessions without
# shrinking the training pool.
#
# These were one variable before. Part B set it to [200, 400] to run only those two
# held-out values, which also cut its training pool to a single magnification: its
# ho=400 runs trained on 200x alone rather than {40,100,200}. That is not LOMO, and it
# is why Part B took 4 h while Part A needed 11.5 h for fewer runs.
MAGNIFICATIONS = [40, 100, 200, 400]        # the pool -- do not narrow this
if RUN_MODE == 'pilot':
    SEEDS = [13]; PATIENT_FRACTIONS = [1.00]; HELD_OUT_MAGS = [400]
elif RUN_MODE == 'quick_12h':
    SEEDS = [13, 42, 2025]; PATIENT_FRACTIONS = [1.00]; HELD_OUT_MAGS = [200, 400]
elif RUN_MODE == 'medium_12h':
    SEEDS = [13, 42, 2025, 3407, 9001]; PATIENT_FRACTIONS = [0.25, 0.50, 1.00]; HELD_OUT_MAGS = [40, 100]
elif RUN_MODE == 'full':
    SEEDS = [13, 42, 2025, 3407, 9001]; PATIENT_FRACTIONS = [0.25, 0.50, 1.00]; HELD_OUT_MAGS = [40, 100, 200, 400]
else:
    raise ValueError(f'Unknown RUN_MODE={RUN_MODE}')
assert set(HELD_OUT_MAGS) <= set(MAGNIFICATIONS), 'HELD_OUT_MAGS must be a subset of the pool'
# With several Kaggle accounts, run ONE held-out magnification per account in parallel so
# each account's run usually fits a single commit and needs no resume. HELD_OUT_OVERRIDE
# replaces the profile's list for this account.
if HELD_OUT_OVERRIDE is not None:
    HELD_OUT_MAGS = [int(m) for m in HELD_OUT_OVERRIDE]
    assert set(HELD_OUT_MAGS) <= set(MAGNIFICATIONS), 'HELD_OUT_OVERRIDE must be drawn from the pool'
# The name is derived from the magnifications so each account's output folder is distinct
# and the resume cell searches for the right folder without a second setting to keep in sync.
if RUN_NAME is None:
    RUN_NAME = 'main_ho' + '-'.join(str(m) for m in HELD_OUT_MAGS)
OUTPUT = f'/kaggle/working/{RUN_NAME}'

PILOT_HELD_OUT = HELD_OUT_MAGS[-1]
PILOT_SEEDS = [SEEDS[0]]
MC_DROPOUT_PASSES = 20
TEMPERATURE_SCALE = True
OOD_METHODS = ['msp', 'energy']
MAX_RUNTIME_HOURS = 11.5
CHECKPOINT_RULE = 'maximum validation macro-F1; earliest epoch breaks ties; test evaluated once after selection'

print(f'RUN_MODE   = {RUN_MODE}')
print(f'RUN_NAME   = {RUN_NAME}')
print(f'OUTPUT     = {OUTPUT}')
print(f'SEEDS      = {SEEDS}')
print(f'FRACTIONS  = {PATIENT_FRACTIONS}')
print(f'POOL       = {MAGNIFICATIONS}   (training draws from pool minus held-out)')
print(f'HELD-OUT   = {HELD_OUT_MAGS}   (this session)')
print(f'MODELS     = {FROM_SCRATCH_MODELS} (+ Phikon if USE_FOUNDATION_MODEL={USE_FOUNDATION_MODEL})')


In [ ]:
# Clear runs left in /kaggle/working for this RUN_NAME, before the resume cell runs.
# On a "Save & Run All" commit /kaggle/working starts empty, so this is a no-op there. It
# matters only in an interactive session with file persistence, where stale runs -- for
# example ones trained before the manifest-regex fix -- would otherwise be skipped as
# already finished. Valid finished runs are restored afterwards by the resume cell, which
# accepts only runs whose manifest fingerprint matches this session's, so leaving this on
# loses nothing that is still valid.
RESET_STATE = True

if RESET_STATE:
    import shutil
    from pathlib import Path
    for pth in [Path(f'/kaggle/working/{RUN_NAME}', 'runs'),
                Path(f'/kaggle/working/{RUN_NAME}', 'raw_runs.csv'),
                Path(f'/kaggle/working/{RUN_NAME}', 'bach_ood_scores.csv'),
                Path(f'/kaggle/working/{RUN_NAME}', 'five_seed_summary.csv'),
                Path(f'/kaggle/working/{RUN_NAME}', 'pareto_scratch.csv'),
                Path(f'/kaggle/working/{RUN_NAME}', 'pilot_records.csv')]:
        if pth.exists():
            shutil.rmtree(pth) if pth.is_dir() else pth.unlink()
            print('deleted', pth)
    print('Local run state cleared. Verified runs from an attached earlier version are '
          'restored by the resume cell.')
else:
    print('RESET_STATE = False; runs already in /kaggle/working are kept and skipped as finished.')


In [ ]:
# Environment. Install only what is missing; do not downgrade Kaggle-provided packages.
import os, sys, json, random, time, hashlib, platform, subprocess, shutil, re
from pathlib import Path
from datetime import datetime, timezone

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
Path(OUTPUT).mkdir(parents=True, exist_ok=True)
Path(SOURCE_ROOT).mkdir(parents=True, exist_ok=True)

def pip_install_if_missing(module_name, spec=None):
    try:
        __import__(module_name); return True
    except ImportError:
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', spec or module_name], check=True)
            return True
        except subprocess.CalledProcessError as exc:
            print(f'Install failed for {spec or module_name}:', exc); return False

for module_name, spec in [
    ('timm', 'timm'),
    ('einops', 'einops'),
    ('transformers', 'transformers'),
    ('huggingface_hub', 'huggingface_hub'),
    ('fvcore', 'fvcore'),
]:
    pip_install_if_missing(module_name, spec)

# Ensure a timm version new enough for MambaVision (timm >= 1.0 exposes mambavision_tiny_1k).
import timm
if int(timm.__version__.split('.')[0]) < 1:
    print(f'timm {timm.__version__} may be too old for MambaVision; upgrading...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm'], check=True)
    import importlib; timm = importlib.reload(timm)
MAMBAVISION_OK = 'mambavision_tiny_1k' in timm.list_models('mambavision*')
print(f'timm = {timm.__version__}    MambaVision available: {MAMBAVISION_OK}')

# Optional compiled Mamba (for MedMamba/DSA-Mamba): try latest, then older pins; skip on failure.
MAMBA_SSM_OK = False
try:
    import mamba_ssm; MAMBA_SSM_OK = True
    print('mamba_ssm already installed:', mamba_ssm.__version__)
except ImportError:
    for spec in [
        ['mamba-ssm', 'causal-conv1d'],
        ['mamba-ssm==2.2.4', 'causal-conv1d==1.5.0.post8'],
        ['mamba-ssm==2.2.2', 'causal-conv1d==1.4.0'],
        ['mamba-ssm==1.2.0.post1', 'causal-conv1d==1.2.0.post2'],
    ]:
        try:
            print('Trying compiled mamba_ssm install:', spec)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation'] + spec, check=True)
            import importlib; mamba_ssm = importlib.import_module('mamba_ssm'); MAMBA_SSM_OK = True
            print(' -> loaded mamba_ssm:', mamba_ssm.__version__); break
        except Exception as exc:
            print(' -> failed:', str(exc)[:180])
    if not MAMBA_SSM_OK:
        print('mamba_ssm unavailable; MedMamba and DSA-Mamba will be skipped. MambaVision through timm still provides the Mamba axis if MAMBAVISION_OK.')

import numpy as np, pandas as pd, torch, torch.nn as nn
import torchvision, sklearn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, brier_score_loss)

assert torch.cuda.is_available(), 'Kaggle GPU accelerator is required.'
DEVICE = torch.device('cuda:0')

def command_text(args):
    try: return subprocess.check_output(args, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as exc: return str(exc)

env = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'torch_cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'gpu_name': torch.cuda.get_device_name(0),
    'gpu_capability': torch.cuda.get_device_capability(0),
    'timm': timm.__version__, 'sklearn': sklearn.__version__,
    'mambavision_ok': MAMBAVISION_OK, 'mamba_ssm_ok': MAMBA_SSM_OK,
    'pip_freeze': command_text([sys.executable, '-m', 'pip', 'freeze']),
    'nvidia_smi': command_text(['nvidia-smi']),
}
Path(OUTPUT, 'environment.json').write_text(json.dumps(env, indent=2, default=str))
config = {k: v for k, v in globals().items()
          if k.isupper() and k not in {'OUTPUT', 'SOURCE_ROOT', 'DEVICE', 'BREAKHIS_ROOT', 'BACH_ROOT'}}
Path(OUTPUT, 'config.json').write_text(json.dumps(config, indent=2, default=str))
print(json.dumps({k: env[k] for k in ('gpu_name','torch','torch_cuda','cudnn','timm','mambavision_ok','mamba_ssm_ok')}, indent=2))


In [ ]:
# Optional MedMamba + DSA-Mamba source pin (only if mamba_ssm compiled). Each Mamba variant is independent.
EXTERNAL_FACTORIES = {}
source_manifest = {}

def clone_at(name, url, commit=None, force_refresh=False):
    destination = Path(SOURCE_ROOT, name)
    if destination.exists() and force_refresh: shutil.rmtree(destination)
    if not destination.exists():
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    if commit is not None:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    return destination

if MAMBA_SSM_OK:
    # --- MedMamba (pinned commit) ---
    try:
        med_src = clone_at('MedMamba', 'https://github.com/YubiaoYue/MedMamba.git',
                           commit='104beb3d641d67cef6f09b94ac299083d91bc417')
        safe_med = Path(SOURCE_ROOT, 'medmamba_safe.py')
        source = Path(med_src, 'MedMamba.py').read_text()
        marker = '\nmedmamba_t = VSSM'
        safe_med.write_text(source.split(marker, 1)[0] + '\n' if marker in source else source)
        sys.path.insert(0, str(Path(SOURCE_ROOT)))
        import importlib.util
        spec = importlib.util.spec_from_file_location('medmamba_safe', safe_med)
        med_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(med_module)
        MedMambaVSSM = med_module.VSSM
        EXTERNAL_FACTORIES['medmamba'] = lambda n: MedMambaVSSM(depths=[2,2,4,2], dims=[96,192,384,768], num_classes=n)
        source_manifest['MedMamba_commit'] = command_text(['git','-C',str(med_src),'rev-parse','HEAD'])
        print('MedMamba registered.')
    except Exception as exc:
        print('MedMamba registration failed:', repr(exc)[:180])

    # --- DSA-Mamba (try pinned, fall back to HEAD) ---
    for attempt_commit in ['7af166288e5e0b710481788c15af79326017b37b', None]:
        try:
            dsa_src = clone_at('DSA-Mamba', 'https://github.com/First-Ronin/DSA-Mamba.git',
                               commit=attempt_commit, force_refresh=(attempt_commit is None))
            sys.path.insert(0, str(dsa_src))
            for mod in list(sys.modules):
                if mod.startswith('model'): del sys.modules[mod]
            from model.DSAmamba import VSSM as DSAMambaVSSM
            EXTERNAL_FACTORIES['dsamamba'] = lambda n: DSAMambaVSSM(in_chans=3, num_classes=n)
            source_manifest['DSA_Mamba_commit'] = command_text(['git','-C',str(dsa_src),'rev-parse','HEAD'])
            source_manifest['DSA_Mamba_attempt'] = 'pinned' if attempt_commit else 'HEAD_fallback'
            print(f'DSA-Mamba registered ({source_manifest["DSA_Mamba_attempt"]}).'); break
        except Exception as exc:
            print(f'DSA-Mamba attempt ({attempt_commit or "HEAD"}) failed:', repr(exc)[:180])
else:
    print('Skipping MedMamba + DSA-Mamba (mamba_ssm not available). MambaVision through timm supplies the Mamba axis.')

if source_manifest:
    Path(OUTPUT, 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2))

# Append registered external Mamba variants to FROM_SCRATCH_MODELS.
for name in ('medmamba', 'dsamamba'):
    if name in EXTERNAL_FACTORIES and name not in FROM_SCRATCH_MODELS:
        FROM_SCRATCH_MODELS.append(name)

# Remove MambaVision from the active list if the installed timm can't build it.
if not MAMBAVISION_OK and 'mambavision_tiny_1k' in FROM_SCRATCH_MODELS:
    FROM_SCRATCH_MODELS.remove('mambavision_tiny_1k')
    print('Removed mambavision_tiny_1k from active models (not in installed timm).')

print('\nActive from-scratch models:', FROM_SCRATCH_MODELS)


In [ ]:
# ---------- Restore DSA-Mamba's missing cross_attention.py and add it to this run ----------
# The sources cell above cannot import DSA-Mamba on either the pinned commit or HEAD:
#     ModuleNotFoundError: No module named 'model.cross_attention'
# model/DSAmamba.py does `from .cross_attention import CrossAttention`, but the upstream
# repository no longer ships that file. It was present in the author's own commit 94c5042
# and went away when the model/ directory was deleted; DSAmamba.py was re-added later and
# cross_attention.py was not. The copy in the only public fork was rewritten over six
# commits and is not the published module, so this cell restores the ORIGINAL file from
# 94c5042 -- recovering deleted upstream code, not reconstructing it.
#
# Verified in the Part C run: builds and runs a forward pass, 43.40M parameters, peak
# memory 264 MiB at batch 2 (~4.2 GiB extrapolated to batch 32).
#
# The sources cell has already appended the registered Mamba variants to
# FROM_SCRATCH_MODELS by the time this cell runs, so a model restored here is added to
# that list explicitly.
TRY_DSAMAMBA_RESTORE = False   # deadline run: DSA-Mamba left out (~4 h per account); True adds it back
DSAMAMBA_CROSS_ATTENTION_COMMIT = '94c5042'      # First-Ronin, 2025-06-09

if not TRY_DSAMAMBA_RESTORE:
    if 'dsamamba' in FROM_SCRATCH_MODELS:
        FROM_SCRATCH_MODELS.remove('dsamamba')
    print('DSA-Mamba not included (TRY_DSAMAMBA_RESTORE = False).')
    print('  Active from-scratch models:', FROM_SCRATCH_MODELS)
elif 'dsamamba' in EXTERNAL_FACTORIES:
    print('DSA-Mamba already registered by the previous cell; nothing to restore.')
    if 'dsamamba' in FROM_SCRATCH_MODELS:
        FROM_SCRATCH_MODELS.remove('dsamamba')
    FROM_SCRATCH_MODELS.insert(0, 'dsamamba')   # train first, see the registered branch below
elif not MAMBA_SSM_OK:
    print('DSA-Mamba restore skipped: mamba_ssm unavailable, so DSAmamba cannot run.')
else:
    import urllib.request, importlib, torch
    DSA_DIR = Path(SOURCE_ROOT, 'DSA-Mamba')
    target = Path(DSA_DIR, 'model', 'cross_attention.py')
    url = ('https://raw.githubusercontent.com/First-Ronin/DSA-Mamba/'
           f'{DSAMAMBA_CROSS_ATTENTION_COMMIT}/model/cross_attention.py')
    try:
        if not DSA_DIR.exists():
            raise RuntimeError(f'{DSA_DIR} not found - the clone in the previous cell '
                               f'must succeed before the file can be restored.')
        if target.exists():
            print(f'cross_attention.py already present at {target}')
        else:
            print(f'Fetching original cross_attention.py from commit '
                  f'{DSAMAMBA_CROSS_ATTENTION_COMMIT} ...')
            with urllib.request.urlopen(url, timeout=60) as resp:
                blob = resp.read()
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(blob)
            print(f'  wrote {target}  ({len(blob)} bytes)')
            # Record provenance next to the file so the origin is never ambiguous later.
            Path(DSA_DIR, 'model', 'cross_attention.PROVENANCE.txt').write_text(
                f'Restored from {url}\n'
                f'Author: First-Ronin (original repo owner)\n'
                f'Commit: {DSAMAMBA_CROSS_ATTENTION_COMMIT} "Add files via upload", 2025-06-09\n'
                f'Reason: a later upstream commit deleted the model directory. DSAmamba.py\n'
                f'        was re-added afterwards, this file was not, yet DSAmamba.py line 12\n'
                f'        still does `from .cross_attention import CrossAttention`.\n'
                f'Verified: the URL above serves this file, and the current upstream model/\n'
                f'          tree contains only DSAmamba.py.\n'
                f'NOT the junaidmaqbool fork version, which was rewritten in 6 commits.\n',
                encoding='utf-8')

        sanity = target.read_text(encoding='utf-8', errors='replace')
        assert 'class CrossAttention' in sanity, \
            'restored file does not define CrossAttention - refusing to use it'
        print(f'  defines: ' + ', '.join(
            sorted({ln.split("(")[0].replace("class ", "").strip()
                    for ln in sanity.splitlines() if ln.startswith("class ")})))

        # Re-import cleanly: drop any half-initialised `model.*` modules from the failed
        # attempt in the previous cell, then import again.
        if str(DSA_DIR) not in sys.path:
            sys.path.insert(0, str(DSA_DIR))
        for mod in [m for m in list(sys.modules) if m == 'model' or m.startswith('model.')]:
            del sys.modules[mod]
        importlib.invalidate_caches()
        from model.DSAmamba import VSSM as DSAMambaVSSM
        print('  import model.DSAmamba OK')

        # Smoke test: build + one forward pass at B=2, reporting peak memory so the
        # B=32 training cost can be extrapolated before committing GPU hours.
        _dev = DEVICE
        m = DSAMambaVSSM(in_chans=3, num_classes=NUM_CLASSES).to(_dev).eval()
        n_params = sum(p.numel() for p in m.parameters())
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            out = m(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=_dev))
        assert tuple(out.shape) == (2, NUM_CLASSES), f'unexpected output {tuple(out.shape)}'
        peak_mib = (torch.cuda.max_memory_allocated() / 2**20) if torch.cuda.is_available() else float('nan')
        print(f'  FORWARD OK  params={n_params/1e6:.2f}M  output={tuple(out.shape)}')
        if peak_mib == peak_mib:
            print(f'  peak GPU memory at B=2: {peak_mib:.0f} MiB'
                  f'  -> rough estimate at B={BATCH_SIZE}: {peak_mib * BATCH_SIZE / 2:.0f} MiB')
            if peak_mib * BATCH_SIZE / 2 > 13000:
                print(f'  WARNING: extrapolated training memory exceeds a T4 (~15 GiB usable).')
                print(f'           Consider a smaller BATCH_SIZE for DSA-Mamba specifically.')
        del m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        EXTERNAL_FACTORIES['dsamamba'] = lambda n: DSAMambaVSSM(in_chans=3, num_classes=n)
        source_manifest['DSA_Mamba_cross_attention_restored_from'] = \
            f'First-Ronin@{DSAMAMBA_CROSS_ATTENTION_COMMIT}'
        try:
            source_manifest['DSA_Mamba_commit'] = command_text(
                ['git', '-C', str(DSA_DIR), 'rev-parse', 'HEAD']).strip()
        except Exception:
            pass
        Path(OUTPUT, 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2))
        print('\nDSA-Mamba REGISTERED and verified.')
        # Front of the list, so DSA-Mamba trains first: it is the only encoder whose training
        # (backward pass, memory at batch 32, wall-clock) has never run on Kaggle, and a
        # problem should surface in the first hour rather than after the other nine runs.
        if 'dsamamba' in FROM_SCRATCH_MODELS:
            FROM_SCRATCH_MODELS.remove('dsamamba')
        FROM_SCRATCH_MODELS.insert(0, 'dsamamba')
        print('  Active from-scratch models:', FROM_SCRATCH_MODELS)
    except Exception as exc:
        print(f'\nDSA-Mamba restore FAILED: {repr(exc)[:400]}')
        print('  Report this as a reproducibility limitation rather than working around it:')
        print('  the published repo is missing model/cross_attention.py, which its core')
        print('  SS_Conv_SSM block imports, so DSA-Mamba cannot be built as published.')


In [ ]:
# BreakHis manifest with patient-disjoint LOMO split (properly stratified per class).
# Slide ids in BreakHis are alphanumeric (22549AB, 23060AB, 16184CD, 3411F), not
# purely numeric. The earlier pattern required -\d+-\d+ here, so build_manifest's
# `if not match: continue` silently discarded every suffixed slide: 7909 files on
# disk became 5792 rows, 82 patients became 59, and because benign slides carry a
# letter suffix far more often, 17 of the 24 benign patients were lost.
NAME = re.compile(r'^(?P<patient>SOB_(?P<coarse>[BM])_(?P<subtype>[A-Z]+)-\d+-[A-Za-z0-9]+)-(?P<magnification>\d+)-\d+', re.I)
SUBTYPE_TO_ID = {'A': 0, 'F': 1, 'PT': 2, 'TA': 3, 'DC': 4, 'LC': 5, 'MC': 6, 'PC': 7}

def discover_breakhis_root(hint=None, search_root='/kaggle/input'):
    candidates = []
    if hint and Path(hint).exists():
        candidates.append(Path(hint))
    if Path(search_root).exists():
        for top in sorted(Path(search_root).iterdir()):
            if top.is_dir():
                candidates.append(top)
        candidates.append(Path(search_root))
    best = None
    best_count = 0
    for candidate in candidates:
        try:
            count = sum(1 for _ in Path(candidate).rglob('SOB_*.png'))
        except (PermissionError, OSError):
            continue
        if count > best_count:
            best = candidate
            best_count = count
    if best is None or best_count == 0:
        raise RuntimeError('Could not auto-discover BreakHis root. Attach ambarish/breakhis or set BREAKHIS_ROOT.')
    print('Auto-discovered BREAKHIS_ROOT =', best, ' (', best_count, 'SOB PNG files)')
    return str(best)

if BREAKHIS_ROOT is None:
    BREAKHIS_ROOT = discover_breakhis_root()
elif not list(Path(BREAKHIS_ROOT).rglob('SOB_*.png')):
    print('Supplied BREAKHIS_ROOT=', BREAKHIS_ROOT, 'has no SOB files; retrying auto-discovery.')
    BREAKHIS_ROOT = discover_breakhis_root()

def build_manifest(root):
    rows = []
    for path in sorted(Path(root).rglob('SOB_*.png')):
        match = NAME.match(path.name)
        if not match:
            continue
        item = match.groupdict()
        subtype = item['subtype'].upper()
        if subtype not in SUBTYPE_TO_ID:
            continue
        rows.append({
            'path': str(path),
            'patient_id': item['patient'],
            'magnification': int(item['magnification']),
            'coarse_label': int(item['coarse'].upper() == 'M'),
            'subtype': subtype,
            'subtype_label': SUBTYPE_TO_ID[subtype],
        })
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise RuntimeError('No matching images found under ' + str(root))
    return frame

def patient_lomo_split(frame, held_out_mag, fraction=1.0, seed=2025,
                       val_ratio=0.15, test_ratio=0.20, val_on_held_out=False):
    """Patient-disjoint LOMO split with PROPER per-class stratification.
    Both benign and malignant patients are proportionally represented in every split
    (train, val, test) to prevent single-class val/test collapse.
    """
    frame = frame.copy()
    patient_class = frame.groupby('patient_id').coarse_label.first().reset_index()
    rng = np.random.default_rng(seed)
    train_patients, val_patients, test_patients = set(), set(), set()
    for cls in sorted(patient_class.coarse_label.unique()):
        group = patient_class.loc[patient_class.coarse_label == cls, 'patient_id'].to_numpy()
        perm = rng.permutation(group)
        n = len(perm)
        n_test = max(1, int(round(test_ratio * n)))
        n_val = max(1, int(round(val_ratio * n)))
        assert n_test + n_val < n, 'Class ' + str(cls) + ' has only ' + str(n) + ' patients; val+test ratios too high.'
        test_patients.update(perm[:n_test].tolist())
        val_patients.update(perm[n_test:n_test + n_val].tolist())
        remain = perm[n_test + n_val:].tolist()
        n_keep = max(1, int(round(fraction * len(remain))))
        train_patients.update(remain[:n_keep])
    train_mags = [m for m in MAGNIFICATIONS if m != held_out_mag]
    frame['split'] = 'drop'
    frame.loc[frame.patient_id.isin(train_patients) & frame.magnification.isin(train_mags), 'split'] = 'train'
    if val_on_held_out:
        frame.loc[frame.patient_id.isin(val_patients) & (frame.magnification == held_out_mag), 'split'] = 'val'
    else:
        frame.loc[frame.patient_id.isin(val_patients) & frame.magnification.isin(train_mags), 'split'] = 'val'
    frame.loc[frame.patient_id.isin(test_patients) & (frame.magnification == held_out_mag), 'split'] = 'test'
    subset = frame.loc[frame.split != 'drop'].copy()
    leak = subset.groupby('patient_id').split.nunique().gt(1).any()
    assert not leak, 'Patient leakage detected.'
    for name in ('train', 'val', 'test'):
        rows = subset[subset.split == name]
        if len(rows) == 0:
            raise RuntimeError('Split ' + name + ' empty (held_out=' + str(held_out_mag) + ', fraction=' + str(fraction) + ', seed=' + str(seed) + ').')
        counts = rows.coarse_label.value_counts().to_dict()
        if len(counts) < 2:
            raise RuntimeError('Split ' + name + ' has only one class ' + str(counts) + ' -- stratification failed.')
    return subset

MANIFEST = build_manifest(BREAKHIS_ROOT)
MANIFEST.to_csv(Path(OUTPUT, 'breakhis_manifest.csv'), index=False)
manifest_hash = hashlib.sha256(MANIFEST.to_csv(index=False).encode()).hexdigest()
Path(OUTPUT, 'breakhis_manifest.sha256').write_text(manifest_hash + chr(10))
print('total images:', len(MANIFEST), '  patients:', MANIFEST.patient_id.nunique())
print('split-manifest SHA-256:', manifest_hash)
print(MANIFEST.groupby(['magnification', 'coarse_label']).size().unstack(fill_value=0))
print('per-patient class counts:', MANIFEST.groupby('patient_id').coarse_label.first().value_counts().to_dict())

_probe = patient_lomo_split(MANIFEST, PILOT_HELD_OUT, fraction=1.0, seed=PILOT_SEEDS[0])
print('Split sanity check at held_out_mag=' + str(PILOT_HELD_OUT) + ':')
print(_probe.groupby(['split', 'magnification']).size().unstack(fill_value=0))
print('patients per split:', _probe.groupby('split').patient_id.nunique().to_dict())
print('class balance per split:')
for name in ('train', 'val', 'test'):
    counts = _probe[_probe.split == name].coarse_label.value_counts().to_dict()
    print('  ' + name + ':', counts)

class HistologyDataset(Dataset):
    def __init__(self, rows, train, mean=(.485, .456, .406), std=(.229, .224, .225)):
        self.rows = rows.reset_index(drop=True)
        ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
        if train:
            ops += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                    transforms.RandomRotation(15), transforms.ColorJitter(.1, .1, .05)]
        self.transform = transforms.Compose(ops + [transforms.ToTensor(),
                                                    transforms.Normalize(mean, std)])
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows.iloc[index]
        y = int(row.coarse_label if TASK == 'binary' else row.subtype_label)
        return self.transform(Image.open(row.path).convert('RGB')), y, row.patient_id


In [ ]:
# ---------- Resume from a previous Kaggle version of this notebook ----------
# Every "Save & Run All" commit starts with an EMPTY /kaggle/working, so the protocol
# cell's record.json skip cannot see runs finished by an earlier commit on its own. A
# commit that stops at the time budget is continued like this:
#   1. Add Input -> Your Work -> this notebook -> the version that stopped
#   2. Save & Run All again
# This cell finds that version's `<RUN_NAME>/runs` folder under /kaggle/input and copies
# the finished runs into OUTPUT before the protocol cell builds its queue.
#
# Runs are copied ONLY if they were trained on exactly this manifest. Outputs from before
# the manifest-regex fix (5792 images / 59 patients) must never leak into a corrected run,
# so a fingerprint of the attached breakhis_manifest.csv is compared with this session's
# MANIFEST first. The fingerprint uses file names rather than absolute paths, because the
# mount point under /kaggle/input can differ between sessions.
import hashlib, shutil
RESUME_FROM_INPUT = True

def _manifest_fingerprint(frame):
    view = pd.DataFrame({
        'file': frame['path'].map(lambda p: Path(str(p)).name),
        'patient_id': frame['patient_id'].astype(str),
        'magnification': frame['magnification'].astype(int),
        'coarse_label': frame['coarse_label'].astype(int),
    }).sort_values(['file', 'magnification', 'patient_id']).reset_index(drop=True)
    return hashlib.sha256(view.to_csv(index=False).encode()).hexdigest()

def _find_previous_outputs(run_name, search_root='/kaggle/input'):
    root = Path(search_root)
    if not root.exists():
        return []
    return sorted({cand for cand in root.rglob(run_name)
                   if cand.is_dir() and (cand / 'runs').is_dir()})

if not RESUME_FROM_INPUT:
    print('Resume disabled (RESUME_FROM_INPUT = False).')
else:
    _here = _manifest_fingerprint(MANIFEST)
    _prev = _find_previous_outputs(RUN_NAME)
    if not _prev:
        print(f'No previous output named {RUN_NAME!r} under /kaggle/input - starting fresh.')
    _resumed = 0
    for _src in _prev:
        _man = _src / 'breakhis_manifest.csv'
        if not _man.exists():
            print(f'  SKIPPED {_src}: no breakhis_manifest.csv, so its runs cannot be verified.')
            continue
        _theirs = pd.read_csv(_man)
        if _manifest_fingerprint(_theirs) != _here:
            print(f'  REFUSED {_src}')
            print(f'    its runs were trained on a different manifest '
                  f'({len(_theirs)} rows, {_theirs.patient_id.nunique()} patients) than this '
                  f'session ({len(MANIFEST)} rows, {MANIFEST.patient_id.nunique()} patients).')
            print('    Most likely an output from before the manifest fix; those runs are retrained.')
            continue
        _n = 0
        # record.json is written only when a run finishes, so an interrupted run (best.pt
        # but no record.json) is not copied and gets retrained.
        for _rec in sorted((_src / 'runs').glob('*/record.json')):
            _dst = Path(OUTPUT, 'runs', _rec.parent.name)
            if (_dst / 'record.json').exists():
                continue
            shutil.copytree(_rec.parent, _dst, dirs_exist_ok=True)
            _n += 1
        _resumed += _n
        print(f'  resumed {_n} finished runs from {_src}')
    if _prev:
        print(f'Total resumed: {_resumed}. The protocol cell skips these and rebuilds raw_runs.csv.')


In [ ]:
# Model factory + parameter/FLOP report.
def build_from_scratch(name, num_classes):
    if name == 'convnext_tiny':
        return timm.create_model('convnext_tiny', pretrained=True, num_classes=num_classes)
    if name == 'swin_tiny':
        return timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
    if name == 'mambavision_tiny_1k':
        # Pretrained weights come from Hugging Face; requires Internet on the first call.
        return timm.create_model('mambavision_tiny_1k', pretrained=True, num_classes=num_classes)
    if name in EXTERNAL_FACTORIES:
        return EXTERNAL_FACTORIES[name](num_classes)
    raise KeyError(f'Unknown or unconfigured model: {name}')

def parameter_and_flop_report(name, model):
    try:
        from fvcore.nn import FlopCountAnalysis, parameter_count
        model.eval().to(DEVICE)
        dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        try: flops = FlopCountAnalysis(model, dummy).total()
        except Exception as exc: flops = float('nan'); print(f'FLOP count skipped for {name}:', str(exc)[:120])
        # parameter_count() is keyed by module prefix and each entry already includes its
        # submodules, so summing every value counted each parameter once per tree level
        # (~7.7x inflation in the previous compute report). The root key '' is the total.
        params = int(parameter_count(model).get('', 0)) or sum(p.numel() for p in model.parameters())
    except Exception as exc:
        params = sum(p.numel() for p in model.parameters()); flops = float('nan')
        print(f'fvcore unavailable for {name}:', str(exc)[:120])
    return {'model': name, 'parameters': int(params),
            'flops_g_at_224': flops / 1e9 if flops == flops else float('nan')}

report = []
for name in list(FROM_SCRATCH_MODELS):
    try:
        m = build_from_scratch(name, NUM_CLASSES)
        report.append(parameter_and_flop_report(name, m))
        del m; torch.cuda.empty_cache()
    except Exception as exc:
        print(f'Build failed for {name}, removing from active list:', repr(exc)[:200])
        if name in FROM_SCRATCH_MODELS: FROM_SCRATCH_MODELS.remove(name)
pd.DataFrame(report).to_csv(Path(OUTPUT, 'from_scratch_compute_report.csv'), index=False)
print('\nActive from-scratch models after build check:', FROM_SCRATCH_MODELS)
pd.DataFrame(report)


In [ ]:
# Frozen foundation-model feature extractor. Phikon is public.
FM_MODEL = None; FM_PROCESSOR = None; FM_DIM = None

def build_foundation_extractor(name):
    from transformers import AutoModel, AutoImageProcessor
    token = os.environ.get('HF_TOKEN', None)
    try:
        processor = AutoImageProcessor.from_pretrained(name, token=token, trust_remote_code=True)
    except Exception:
        processor = None
    model = AutoModel.from_pretrained(name, token=token, trust_remote_code=True).eval().to(DEVICE)
    for parameter in model.parameters(): parameter.requires_grad_(False)
    return model, processor

class LinearProbe(nn.Module):
    def __init__(self, feature_dim, num_classes):
        super().__init__(); self.head = nn.Linear(feature_dim, num_classes)
    def forward(self, x): return self.head(x)

@torch.inference_mode()
def extract_features(model, loader):
    feats, labels, patients = [], [], []
    for x, y, pid in loader:
        x = x.to(DEVICE, non_blocking=True)
        try: out = model(pixel_values=x)
        except TypeError: out = model(x)
        if hasattr(out, 'last_hidden_state'):
            token = out.last_hidden_state[:, 0]
        elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
            token = out.pooler_output
        else:
            token = out if out.dim() == 2 else out.mean(dim=list(range(1, out.dim() - 1)))
        feats.append(token.detach().cpu()); labels.append(torch.as_tensor(y)); patients.extend(pid)
    return torch.cat(feats), torch.cat(labels), patients

if USE_FOUNDATION_MODEL:
    try:
        FM_MODEL, FM_PROCESSOR = build_foundation_extractor(FM_NAME)
        with torch.inference_mode():
            dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            try: out = FM_MODEL(pixel_values=dummy)
            except TypeError: out = FM_MODEL(dummy)
            if hasattr(out, 'last_hidden_state'):
                FM_DIM = out.last_hidden_state.shape[-1]
            elif hasattr(out, 'pooler_output') and out.pooler_output is not None:
                FM_DIM = out.pooler_output.shape[-1]
            else:
                FM_DIM = out.shape[-1] if out.dim() == 2 else out.mean(dim=list(range(1, out.dim() - 1))).shape[-1]
        print(f'Foundation model {FM_NAME} loaded; feature dim = {FM_DIM}')
    except Exception as exc:
        print(f'Foundation model load failed: {repr(exc)[:200]}. Continuing without the FM branch.')
        USE_FOUNDATION_MODEL = False


In [ ]:
# Calibration and selective-prediction utilities.
# numpy 2.x deprecated np.trapz in favour of np.trapezoid; the old name emitted a
# DeprecationWarning on every metric computation and buried the run log. The two
# are numerically identical, so this alias changes no results.
_trapz = getattr(np, 'trapezoid', None) or np.trapz

def expected_calibration_error(probs, labels, n_bins=15):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lower, upper in zip(bins[:-1], bins[1:]):
        in_bin = (confidences > lower) & (confidences <= upper)
        if in_bin.sum() == 0: continue
        gap = abs(accuracies[in_bin].mean() - confidences[in_bin].mean())
        ece += in_bin.mean() * gap
    return float(ece)

def brier_multiclass(probs, labels):
    one_hot = np.eye(probs.shape[1])[labels]
    return float(((probs - one_hot) ** 2).sum(axis=1).mean())

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__(); self.log_temp = nn.Parameter(torch.zeros(1))
    def forward(self, logits): return logits / torch.exp(self.log_temp)

def fit_temperature(val_logits, val_labels, max_iter=200):
    scaler = TemperatureScaler().to(DEVICE)
    logits = torch.as_tensor(val_logits, device=DEVICE, dtype=torch.float32)
    labels = torch.as_tensor(val_labels, device=DEVICE, dtype=torch.long)
    optim = torch.optim.LBFGS([scaler.log_temp], lr=0.1, max_iter=max_iter)
    loss_fn = nn.CrossEntropyLoss()
    def closure():
        optim.zero_grad(); loss = loss_fn(scaler(logits), labels); loss.backward(); return loss
    optim.step(closure)
    return float(torch.exp(scaler.log_temp).item())

def risk_coverage_curve(probs, labels):
    confidences = probs.max(axis=1); predictions = probs.argmax(axis=1)
    order = np.argsort(-confidences)
    errors = (predictions != labels).astype(float)[order]
    coverage = np.arange(1, len(order) + 1) / len(order)
    risk = np.cumsum(errors) / np.arange(1, len(order) + 1)
    return coverage, risk, float(_trapz(risk, coverage))

def maximum_softmax_probability(probs): return probs.max(axis=1)
def energy_score(logits, T=1.0):
    logits = np.asarray(logits, dtype=np.float64)
    return T * (np.log(np.exp(logits / T).sum(axis=1) + 1e-12))

print('Calibration and selective-prediction utilities ready.')


In [ ]:
# Training + evaluation primitives.
STANDARD_RECORD_COLUMNS = ['run_id', 'branch', 'model', 'held_out_mag', 'patient_fraction', 'seed',
                            'peak_memory_mib', 'wallclock_s',
                            'test_accuracy', 'test_balanced_accuracy', 'test_macro_f1',
                            'test_ece', 'test_brier', 'test_auc',
                            'test_risk_coverage_auc', 'test_risk_at_coverage_80',
                            'test_risk_at_coverage_90', 'test_risk_at_coverage_95', 'test_temperature',
                            'error']

def normalize_record(record):
    """Ensure every record has the same column set so incremental CSV append does not misalign columns."""
    for key in STANDARD_RECORD_COLUMNS:
        record.setdefault(key, float('nan'))
    return {key: record.get(key) for key in STANDARD_RECORD_COLUMNS}

def seed_everything(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic

def make_loaders(train_rows, val_rows, test_rows):
    return {
        'train': DataLoader(HistologyDataset(train_rows, True), batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True),
        'val': DataLoader(HistologyDataset(val_rows, False), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True),
        'test': DataLoader(HistologyDataset(test_rows, False), batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True),
    }

@torch.inference_mode()
def logits_and_labels(model, loader):
    model.eval()
    logits_all, labels_all = [], []
    for x, y, _ in loader:
        logits_all.append(model(x.to(DEVICE, non_blocking=True)).detach().cpu().numpy())
        labels_all.append(np.asarray(y))
    return np.concatenate(logits_all), np.concatenate(labels_all)

def compute_all_metrics(logits, labels, temperature=1.0):
    scaled = logits / temperature
    exp = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    probs = exp / exp.sum(axis=1, keepdims=True)
    predictions = probs.argmax(axis=1)
    metrics = {
        'accuracy': accuracy_score(labels, predictions),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
        'ece': expected_calibration_error(probs, labels),
        'brier': brier_multiclass(probs, labels),
    }
    try:
        if NUM_CLASSES == 2:
            metrics['auc'] = roc_auc_score(labels, probs[:, 1])
        else:
            metrics['auc'] = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except ValueError:
        metrics['auc'] = float('nan')
    coverage, risk, rcauc = risk_coverage_curve(probs, labels)
    metrics['risk_coverage_auc'] = rcauc
    for target in (0.80, 0.90, 0.95):
        idx = int(target * len(coverage)) - 1
        metrics['risk_at_coverage_' + str(int(100 * target))] = float(risk[idx]) if idx >= 0 else float('nan')
    metrics['temperature'] = float(temperature)
    return metrics, probs

def train_from_scratch(model_name, held_out_mag, patient_fraction, seed):
    seed_everything(seed)
    subset = patient_lomo_split(MANIFEST, held_out_mag, fraction=patient_fraction, seed=seed)
    loaders = make_loaders(subset[subset.split == 'train'], subset[subset.split == 'val'], subset[subset.split == 'test'])
    model = build_from_scratch(model_name, NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.CrossEntropyLoss()
    best_score = -1.0
    run_id = 'scratch_' + model_name + '_ho' + str(held_out_mag) + '_frac' + str(patient_fraction) + '_seed' + str(seed)
    run_dir = Path(OUTPUT, 'runs', run_id)
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    epochs_log = []
    best_ckpt = run_dir / 'best.pt'
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for x, y, _ in loaders['train']:
            optim.zero_grad(set_to_none=True)
            loss = loss_fn(model(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True))
            loss.backward()
            optim.step()
            running += float(loss.detach()) * len(y)
        val_logits, val_labels = logits_and_labels(model, loaders['val'])
        val_metrics, _ = compute_all_metrics(val_logits, val_labels)
        epoch_entry = {'epoch': epoch, 'train_loss': running / len(loaders['train'].dataset)}
        for k, v in val_metrics.items():
            epoch_entry['val_' + k] = v
        epochs_log.append(epoch_entry)
        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            torch.save(model.state_dict(), best_ckpt)
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    val_logits, val_labels = logits_and_labels(model, loaders['val'])
    temperature = fit_temperature(val_logits, val_labels) if TEMPERATURE_SCALE else 1.0
    test_logits, test_labels = logits_and_labels(model, loaders['test'])
    test_metrics, test_probs = compute_all_metrics(test_logits, test_labels, temperature)
    record = normalize_record({
        'run_id': run_id, 'branch': 'from_scratch', 'model': model_name,
        'held_out_mag': held_out_mag, 'patient_fraction': patient_fraction, 'seed': seed,
        'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
        'wallclock_s': time.perf_counter() - start,
        **{'test_' + k: v for k, v in test_metrics.items()},
    })
    pd.DataFrame(epochs_log).to_csv(run_dir / 'epochs.csv', index=False)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=test_labels, probs=test_probs)
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))
    del model
    torch.cuda.empty_cache()
    return record

def train_fm_linear_probe(held_out_mag, patient_fraction, seed):
    if not USE_FOUNDATION_MODEL:
        return None
    seed_everything(seed)
    subset = patient_lomo_split(MANIFEST, held_out_mag, fraction=patient_fraction, seed=seed)
    loaders = make_loaders(subset[subset.split == 'train'], subset[subset.split == 'val'], subset[subset.split == 'test'])
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    tr_feats, tr_labels, _ = extract_features(FM_MODEL, loaders['train'])
    va_feats, va_labels, _ = extract_features(FM_MODEL, loaders['val'])
    te_feats, te_labels, _ = extract_features(FM_MODEL, loaders['test'])
    probe = LinearProbe(FM_DIM, NUM_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    best_score = -1.0
    best_state = None
    for epoch in range(200):
        probe.train()
        perm = torch.randperm(len(tr_feats))
        for start_idx in range(0, len(perm), 256):
            idx = perm[start_idx:start_idx + 256]
            xb = tr_feats[idx].to(DEVICE)
            yb = tr_labels[idx].to(DEVICE, dtype=torch.long)
            optim.zero_grad(set_to_none=True)
            loss_fn(probe(xb), yb).backward()
            optim.step()
        probe.eval()
        with torch.inference_mode():
            val_logits = probe(va_feats.to(DEVICE)).cpu().numpy()
        val_metrics, _ = compute_all_metrics(val_logits, va_labels.numpy())
        if val_metrics['macro_f1'] > best_score:
            best_score = val_metrics['macro_f1']
            best_state = {k: v.detach().clone() for k, v in probe.state_dict().items()}
    probe.load_state_dict(best_state)
    probe.eval()
    with torch.inference_mode():
        val_logits = probe(va_feats.to(DEVICE)).cpu().numpy()
        test_logits = probe(te_feats.to(DEVICE)).cpu().numpy()
    temperature = fit_temperature(val_logits, va_labels.numpy()) if TEMPERATURE_SCALE else 1.0
    test_metrics, test_probs = compute_all_metrics(test_logits, te_labels.numpy(), temperature)
    run_id = 'fm_' + FM_NAME.replace('/', '_') + '_ho' + str(held_out_mag) + '_frac' + str(patient_fraction) + '_seed' + str(seed)
    run_dir = Path(OUTPUT, 'runs', run_id)
    run_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(run_dir / 'test_outputs.npz', logits=test_logits, labels=te_labels.numpy(), probs=test_probs)
    # Probe head kept for the BACH OOD cell; saved before record.json marks the run finished.
    torch.save({'state_dict': {k: v.cpu() for k, v in best_state.items()}, 'feature_dim': int(FM_DIM),
                'num_classes': NUM_CLASSES, 'fm_name': FM_NAME, 'temperature': float(temperature)},
               run_dir / 'probe.pt')
    record = normalize_record({
        'run_id': run_id, 'branch': 'fm_linear_probe', 'model': FM_NAME,
        'held_out_mag': held_out_mag, 'patient_fraction': patient_fraction, 'seed': seed,
        'peak_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
        'wallclock_s': time.perf_counter() - start,
        **{'test_' + k: v for k, v in test_metrics.items()},
    })
    Path(run_dir, 'record.json').write_text(json.dumps(record, indent=2, default=str))
    return record

print('Training primitives ready. STANDARD_RECORD_COLUMNS enforced.')


In [ ]:
# Feasibility pilot: only runs when RUN_MODE == 'pilot'. Skipped otherwise.
if RUN_MODE == 'pilot':
    pilot_records = []
    for model_name in FROM_SCRATCH_MODELS:
        for seed in PILOT_SEEDS:
            try:
                pilot_records.append(train_from_scratch(model_name, PILOT_HELD_OUT, 1.0, seed))
            except Exception as exc:
                pilot_records.append({'model': model_name, 'seed': seed, 'error': repr(exc)[:400]})
    if USE_FOUNDATION_MODEL:
        for seed in PILOT_SEEDS:
            try:
                pilot_records.append(train_fm_linear_probe(PILOT_HELD_OUT, 1.0, seed))
            except Exception as exc:
                pilot_records.append({'model': FM_NAME, 'seed': seed, 'error': repr(exc)[:400]})
    pilot_frame = pd.DataFrame([r for r in pilot_records if r is not None])
    pilot_frame.to_csv(Path(OUTPUT, 'pilot_records.csv'), index=False)
    print(pilot_frame)
    raise SystemExit('Pilot complete. Inspect pilot_records.csv, environment.json, source_manifest.json. Switch RUN_MODE (e.g. to "quick_12h") to proceed to the real protocol.')
else:
    print(f'Skipping pilot (RUN_MODE={RUN_MODE}); proceeding directly to cell-11 full protocol.')


In [ ]:
# Full protocol under the current RUN_MODE.
# Resumable: skips runs whose record.json already exists.
# Incremental: appends each finished record to raw_runs.csv immediately, with columns pinned.
# Time-budgeted: a run starts only if it should finish in time (see the time guard below).
raw_runs_csv = Path(OUTPUT, 'raw_runs.csv')

def _run_exists(run_id):
    return Path(OUTPUT, 'runs', run_id, 'record.json').exists()

def _load_record(run_id):
    return json.loads(Path(OUTPUT, 'runs', run_id, 'record.json').read_text())

def _append_row(record):
    record = normalize_record(dict(record))
    row = pd.DataFrame([record], columns=STANDARD_RECORD_COLUMNS)
    header = not raw_runs_csv.exists()
    row.to_csv(raw_runs_csv, mode='a', header=header, index=False)

planned = []
for held_out in HELD_OUT_MAGS:
    for fraction in PATIENT_FRACTIONS:
        for model_name in FROM_SCRATCH_MODELS:
            for seed in SEEDS:
                planned.append(('from_scratch', model_name, held_out, fraction, seed,
                                'scratch_' + model_name + '_ho' + str(held_out) + '_frac' + str(fraction) + '_seed' + str(seed)))
        if USE_FOUNDATION_MODEL:
            for seed in SEEDS:
                planned.append(('fm_linear_probe', FM_NAME, held_out, fraction, seed,
                                'fm_' + FM_NAME.replace('/', '_') + '_ho' + str(held_out) + '_frac' + str(fraction) + '_seed' + str(seed)))

print('Total planned runs:', len(planned))
completed = [entry for entry in planned if _run_exists(entry[5])]
remaining = [entry for entry in planned if not _run_exists(entry[5])]
print('Already completed on disk:', len(completed), '   Remaining:', len(remaining))

if completed:
    if raw_runs_csv.exists():
        raw_runs_csv.unlink()
    for entry in completed:
        _append_row(_load_record(entry[5]))
    print('Rebuilt raw_runs.csv from', len(completed), 'existing records.')

# ---- Time guard ----
# Kaggle stops a commit 12 h after its kernel starts, setup cells included, and nothing here
# interrupts a run already under way. So a run is started only if it should finish within
# MAX_RUNTIME_HOURS of kernel start, leaving POST_LOOP_RESERVE_H for the BACH and aggregate
# cells. Expected length = 1.1x the longest finished run of the same model (this commit or
# resumed records), else DEFAULT_RUN_HOURS. A run that does not fit is left for the next
# commit; shorter runs further down the queue still get their chance.
POST_LOOP_RESERVE_H = 0.5
DEFAULT_RUN_HOURS = {'dsamamba': 1.75, 'medmamba': 1.25, 'convnext_tiny': 1.0, 'swin_tiny': 1.0, FM_NAME: 0.1}
try:
    import psutil
    KERNEL_START = psutil.Process(os.getpid()).create_time()
except Exception:
    KERNEL_START = time.time()
    print('psutil unavailable: the time budget counts from this cell, not from kernel start.')
deadline = KERNEL_START + (MAX_RUNTIME_HOURS - POST_LOOP_RESERVE_H) * 3600
print(f'Setup cells took {(time.time() - KERNEL_START) / 60:.1f} min; '
      f'runs must finish within the next {(deadline - time.time()) / 3600:.2f} h.')

longest_run_s = {}
def _note_duration(record):
    seconds = (record or {}).get('wallclock_s')
    if isinstance(seconds, (int, float)) and seconds == seconds and seconds > 0:
        longest_run_s[record['model']] = max(longest_run_s.get(record['model'], 0.0), float(seconds))

def _expected_run_s(model_name):
    if model_name in longest_run_s:
        return 1.1 * longest_run_s[model_name]
    return DEFAULT_RUN_HOURS.get(model_name, 1.75) * 3600

def _fits(model_name):
    return time.time() + _expected_run_s(model_name) <= deadline

for entry in completed:
    _note_duration(_load_record(entry[5]))

session_start = time.perf_counter()
left_for_next_commit = []

for idx, (branch, model_name, held_out, fraction, seed, run_id) in enumerate(remaining, 1):
    elapsed = time.perf_counter() - session_start
    if not _fits(model_name):
        left_for_next_commit.append(run_id)
        print(f'[{idx}/{len(remaining)}] not started: {run_id} needs ~{_expected_run_s(model_name)/3600:.2f} h, '
              f'{max(deadline - time.time(), 0)/3600:.2f} h left')
        continue
    print('[' + str(idx) + '/' + str(len(remaining)) + '] elapsed', round(elapsed/3600, 2), 'h  ',
          branch, model_name, 'ho=' + str(held_out), 'frac=' + str(fraction), 'seed=' + str(seed))
    try:
        if branch == 'from_scratch':
            record = train_from_scratch(model_name, held_out, fraction, seed)
        else:
            record = train_fm_linear_probe(held_out, fraction, seed)
        if record is not None:
            _append_row(record)
            _note_duration(record)
    except Exception as exc:
        _append_row({
            'run_id': run_id, 'branch': branch, 'model': model_name,
            'held_out_mag': held_out, 'patient_fraction': fraction, 'seed': seed,
            'error': repr(exc)[:400],
        })
        print('  -> FAILED:', repr(exc)[:200])

print('Session finished. Total wall-clock:', round((time.perf_counter() - session_start) / 3600, 2), 'h')
if left_for_next_commit:
    print(f'Time budget reached: {len(left_for_next_commit)} run(s) left for the next commit.')
    print('Attach this version as input and Save & Run All again (see the resume cell).')
if raw_runs_csv.exists():
    print('raw_runs.csv rows so far:', len(pd.read_csv(raw_runs_csv)))


In [ ]:
# BACH as OOD test. Auto-discovers BACH_ROOT and scores every finished run on disk: the
# from-scratch checkpoints (best.pt) and the frozen FM's linear-probe heads (probe.pt).
# BACH features are extracted from the FM once and shared by all of its probe heads.
def discover_bach_root(hint=None, search_root='/kaggle/input'):
    candidates = []
    if hint and Path(hint).exists(): candidates.append(Path(hint))
    if Path(search_root).exists():
        for root, dirs, _files in os.walk(search_root):
            if {'Normal','Benign','InSitu','Invasive'}.issubset(set(dirs)):
                candidates.append(Path(root))
    best = None; best_count = 0
    for candidate in set(candidates):
        count = sum(1 for _ in Path(candidate).rglob('*.tif')) + sum(1 for _ in Path(candidate).rglob('*.png'))
        if count > best_count: best = candidate; best_count = count
    if best is None:
        print('BACH root not found. OOD test will be skipped.'); return None
    print(f'Auto-discovered BACH_ROOT = {best}  ({best_count} image files)')
    return str(best)

def bach_ood_row(record, logits):
    logits = np.asarray(logits, dtype=np.float64)
    exp = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = exp / exp.sum(axis=1, keepdims=True)
    msp = maximum_softmax_probability(probs)
    energy = energy_score(logits)
    return {'run_id': record['run_id'], 'branch': record['branch'],
            'model': record['model'], 'held_out_mag': record['held_out_mag'], 'seed': record['seed'],
            'msp_mean': float(msp.mean()), 'msp_std': float(msp.std()),
            'energy_mean': float(energy.mean()), 'energy_std': float(energy.std())}

if BACH_ROOT is None: BACH_ROOT = discover_bach_root()
raw_runs_csv = Path(OUTPUT, 'raw_runs.csv')

if BACH_ROOT is not None and raw_runs_csv.exists():
    def build_bach_manifest(root):
        rows = []
        for cls, cls_id in {'Normal':0,'Benign':1,'InSitu':2,'Invasive':3}.items():
            for suffix in ('*.tif', '*.png', '*.jpg'):
                for path in sorted(Path(root, cls).glob(suffix)):
                    rows.append({'path': str(path), 'bach_class': cls_id})
        return pd.DataFrame(rows)
    try:
        bach_frame = build_bach_manifest(BACH_ROOT)
        if bach_frame.empty:
            print('BACH manifest empty at', BACH_ROOT)
        else:
            class BachDataset(Dataset):
                def __init__(self, rows):
                    self.rows = rows.reset_index(drop=True)
                    self.transform = transforms.Compose([
                        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        transforms.ToTensor(),
                        transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
                def __len__(self): return len(self.rows)
                def __getitem__(self, index):
                    row = self.rows.iloc[index]
                    return self.transform(Image.open(row.path).convert('RGB')), int(row.bach_class), 'bach'
            bach_loader = DataLoader(BachDataset(bach_frame), batch_size=BATCH_SIZE, shuffle=False,
                                     num_workers=NUM_WORKERS, pin_memory=True)
            raw_frame = pd.read_csv(raw_runs_csv)
            if 'error' in raw_frame.columns:
                raw_frame = raw_frame[raw_frame.error.isna()]
            scratch = raw_frame[raw_frame.branch.eq('from_scratch')]
            probes = raw_frame[raw_frame.branch.eq('fm_linear_probe')]
            ood_rows = []
            for _, record in scratch.iterrows():
                ckpt_path = Path(OUTPUT, 'runs', record['run_id'], 'best.pt')
                if not ckpt_path.exists(): continue
                try:
                    model = build_from_scratch(record['model'], NUM_CLASSES).to(DEVICE)
                    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)); model.eval()
                    bach_logits, _ = logits_and_labels(model, bach_loader)
                    ood_rows.append(bach_ood_row(record, bach_logits))
                    del model; torch.cuda.empty_cache()
                except Exception as exc:
                    print(f'  BACH OOD failed for {record["run_id"]}:', repr(exc)[:200])
            if len(probes) and FM_MODEL is None:
                print(f'  {len(probes)} FM probe runs not scored: the foundation model is not loaded.')
            elif len(probes):
                try:
                    bach_feats, _, _ = extract_features(FM_MODEL, bach_loader)
                    for _, record in probes.iterrows():
                        probe_path = Path(OUTPUT, 'runs', record['run_id'], 'probe.pt')
                        if not probe_path.exists():
                            print(f'  {record["run_id"]}: no probe.pt, not scored'); continue
                        saved = torch.load(probe_path, map_location=DEVICE)
                        probe = LinearProbe(saved['feature_dim'], saved['num_classes']).to(DEVICE)
                        probe.load_state_dict(saved['state_dict']); probe.eval()
                        with torch.inference_mode():
                            bach_logits = probe(bach_feats.to(DEVICE)).cpu().numpy()
                        ood_rows.append(bach_ood_row(record, bach_logits))
                except Exception as exc:
                    print('  BACH OOD failed for the FM probes:', repr(exc)[:200])
            if ood_rows:
                pd.DataFrame(ood_rows).to_csv(Path(OUTPUT, 'bach_ood_scores.csv'), index=False)
                n_fm = sum(row['branch'] == 'fm_linear_probe' for row in ood_rows)
                print(f'BACH OOD scores saved: {len(ood_rows)} rows '
                      f'({len(ood_rows) - n_fm} from-scratch, {n_fm} FM probe).')
            else:
                print('BACH OOD: no run could be scored; bach_ood_scores.csv not written.')
    except Exception as exc:
        print(f'BACH OOD stage skipped: {repr(exc)[:200]}')
else:
    print('BACH OOD skipped (BACH root not found or raw_runs.csv missing).')


In [ ]:
# Aggregation. Folder-only mode by default -- no ZIP (saves ~2x space on Kaggle output).
# Set CREATE_ZIP = True below if you want the .zip archive too (roughly doubles the total).
CREATE_ZIP = False

raw_runs_csv = Path(OUTPUT, 'raw_runs.csv')
raw_frame = pd.read_csv(raw_runs_csv) if raw_runs_csv.exists() else pd.DataFrame()
if not raw_frame.empty:
    ok = raw_frame[raw_frame.error.isna()] if 'error' in raw_frame.columns else raw_frame
    metric_cols = [c for c in ok.columns if c.startswith('test_')]
    if metric_cols:
        summary = (ok.groupby(['branch', 'model', 'held_out_mag', 'patient_fraction'])[metric_cols]
                     .agg(['mean', 'std', 'count']))
        summary.to_csv(Path(OUTPUT, 'five_seed_summary.csv'))
    compute = pd.read_csv(Path(OUTPUT, 'from_scratch_compute_report.csv'))
    if 'branch' in ok.columns:
        from_scratch_wide = ok[ok.branch == 'from_scratch'].merge(compute, on='model', how='left')
    else:
        from_scratch_wide = ok
    from_scratch_wide.to_csv(Path(OUTPUT, 'pareto_scratch.csv'), index=False)

print('Artifacts under', OUTPUT, ':')
total_bytes = 0
for name in ['config.json', 'environment.json', 'source_manifest.json', 'breakhis_manifest.csv',
             'breakhis_manifest.sha256', 'from_scratch_compute_report.csv', 'pilot_records.csv',
             'raw_runs.csv', 'five_seed_summary.csv', 'pareto_scratch.csv', 'bach_ood_scores.csv']:
    path = Path(OUTPUT, name)
    if path.exists():
        sz = path.stat().st_size
        total_bytes += sz
        print('  -', name.ljust(40), str(sz).rjust(12), 'bytes')

runs_dir = Path(OUTPUT, 'runs')
if runs_dir.exists():
    run_folders = sorted([p for p in runs_dir.iterdir() if p.is_dir()])
    per_run_bytes = 0
    for rf in run_folders:
        per_run_bytes += sum(f.stat().st_size for f in rf.rglob('*') if f.is_file())
    print('  runs/', len(run_folders), 'runs,', round(per_run_bytes / 1e9, 2), 'GB (best.pt + epochs.csv + test_outputs.npz + record.json per run)')
    total_bytes += per_run_bytes

print('Total output size:', round(total_bytes / 1e9, 2), 'GB')

if CREATE_ZIP:
    print('Creating ZIP archive...')
    archive = shutil.make_archive('/kaggle/working/' + RUN_NAME, 'zip', OUTPUT)
    print('Archive:', archive)
else:
    print('CREATE_ZIP=False -> no zip written. Analysis notebook can read', OUTPUT, 'directly by attaching this notebook output as input.')
    print('To make a zip anyway, set CREATE_ZIP=True and rerun this cell.')
